# Prompted instrument segmentation walkthrough

1. render image/mask pairs from the endoscopic phantom
2. turn a mask into a SAM-style prompt (a click, or a box)
3. build the segmenter, inject LoRA, and see what is trainable
4. train with Dice + BCE, using the trainer's own loop
5. score on a held-out split, with the population named
6. save the adapters and prove the reload is exact

NOTE: `sam2` is not installed in this repo, so this notebook runs `TinyUNetSegmenter`.

## 0. Setup

In [ ]:
import os
import sys
import tempfile
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
os.environ["PSAI_FORCE_CPU"] = "1"

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle
from torch.utils.data import DataLoader

from src.common.device import device_report, get_device
from src.common.seed import seed_everything

plt.rcParams["figure.dpi"] = 58
seed_everything(0)
device = get_device()
print(device_report(device))

ROOT = Path(tempfile.mkdtemp(prefix="psai-seg-nb-"))
print("scratch data root:", ROOT)

## 1. Data

In [ ]:
from src.data.make_dummy_data import make_seg
from src.data.splits import load_splits

make_seg(ROOT / "seg", n_clips=10, n_frames=8, size=64, seed=0)
splits = load_splits(ROOT / "seg")
print()
for name, ids in splits.items():
    print(f"{name:5s} {ids}")
print("train n val:", set(splits["train"]) & set(splits["val"]))
print("train n test:", set(splits["train"]) & set(splits["test"]))

In [ ]:
import json

from PIL import Image

from src.data.phantom import best_threshold_dice  # the same leak detector the tests use

pairs = json.loads((ROOT / "seg" / "pairs.json").read_text())
fig, axes = plt.subplots(2, 4, figsize=(11, 5.4))
leaks = []
for i, p in enumerate(pairs[:4]):
    img = Image.open(ROOT / "seg" / p["image"])
    msk = np.asarray(Image.open(ROOT / "seg" / p["mask"])) > 127
    leaks.append(best_threshold_dice(np.asarray(img.convert("L")), msk))
    axes[0, i].imshow(np.asarray(img)); axes[0, i].set_title(p["video_id"], fontsize=9)
    axes[1, i].imshow(msk, cmap="gray"); axes[1, i].set_title(f"mask ({msk.mean():.1%})", fontsize=9)
for ax in axes.ravel():
    ax.axis("off")
plt.tight_layout()
plt.show()

print("best single-threshold Dice per frame:", [round(v, 2) for v in leaks])
print("(painting the mask into the image, as `images += masks * 2.0` did, scores ~1.0 here)")

## 2. Mask -> prompt

In [ ]:
from src.common.seed import derive_seed, make_generator
from src.segmentation.dataset import MaskDataset, collate, prompt_from_mask, to_display

train_ds = MaskDataset(str(ROOT / "seg"), split=splits["train"], img_size=64,
                       prompt_kind="point", seed=0, name="train")
val_ds = MaskDataset(str(ROOT / "seg"), split=splits["val"], img_size=64,
                     prompt_kind="point", seed=0, name="val")

sample = train_ds[0]
print()
for k, v in sample.items():
    print(f"  {k:14s} {tuple(v.shape) if torch.is_tensor(v) else v}")

mask = sample["mask"][0]
gen = make_generator(derive_seed(0, "train", 0))
pc, pl, _ = prompt_from_mask(mask, "point", gen)
_, _, box = prompt_from_mask(mask, "box", make_generator(0))
print(f"\npoint {pc.tolist()} label {pl.tolist()} | box {box.tolist()}")

try:
    prompt_from_mask(torch.zeros_like(mask), "point", make_generator(0))
except ValueError as exc:
    print("empty mask ->", exc)
print("prompt is reproducible:", torch.equal(train_ds[0]["point_coords"], train_ds[0]["point_coords"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4.0))
axes[0].imshow(to_display(sample["image"])); axes[0].set_title("image (denormalised)")
axes[1].imshow(mask, cmap="gray"); axes[1].set_title("mask, with both prompt kinds")
axes[1].scatter(*pc[0].tolist(), c="lime", s=70, marker="*")
x0, y0, x1, y1 = box.tolist()
axes[1].add_patch(Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, ec="cyan", lw=1.5))
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3. The segmenter, and the prompt channel

In [ ]:
from src.common.config import SegConfig
from src.common.lora import count_trainable
from src.segmentation.sam2_lora import build_segmenter, prompt_channel

cfg = SegConfig(data_root=str(ROOT / "seg"), model="tinyunet", splits="splits.json",
                img_size=64, prompt_kind="point", lora_rank=4, lora_alpha=8,
                lr=1.0e-3, epochs=8, batch_size=4, workers=0, seed=0)
model = build_segmenter(cfg, device)

trainable, total = count_trainable(model)
print(f"backend {model.backend} | LoRA spec {model.lora_spec}")
print(f"trainable {trainable:,} / {total:,} params ({100 * trainable / total:.1f}%)")

shape = torch.Size((1, 3, 64, 64))
left = prompt_channel(shape, torch.tensor([[[16.0, 16.0]]]), torch.tensor([[1]]), None)
right = prompt_channel(shape, torch.tensor([[[48.0, 48.0]]]), torch.tensor([[1]]), None)
pad = prompt_channel(shape, torch.tensor([[[32.0, 32.0]]]), torch.tensor([[-1]]), None)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
for ax, ch, t in zip(axes, (left, right, pad),
                     ("click at (16,16)", "click at (48,48)", "label -1 padding point")):
    ax.imshow(ch[0, 0], cmap="coolwarm", vmin=-1, vmax=1); ax.set_title(t, fontsize=9); ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# the prompt must change the output, or none of the prompt plumbing is tested
img = val_ds[0]["image"].unsqueeze(0).to(device)
model.eval()
with torch.no_grad():
    a = model(img, point_coords=torch.tensor([[[16.0, 16.0]]], device=device),
              point_labels=torch.tensor([[1]], device=device))
    b = model(img, point_coords=torch.tensor([[[48.0, 48.0]]], device=device),
              point_labels=torch.tensor([[1]], device=device))
print("max |logit difference| between two prompts 45px apart:", float((a - b).abs().max()))

try:
    model(img)
except ValueError as exc:
    print("no prompt at all ->", exc)

## 4. The SAM 2 backend

Same interface, same training loop.

In [ ]:
sam2_cfg = SegConfig(data_root=str(ROOT / "seg"), model="sam2",
                     checkpoint="checkpoints/sam2.1_hiera_large.pt",
                     model_cfg="configs/sam2.1/sam2.1_hiera_l.yaml",
                     splits="splits.json", img_size=1024)
try:
    build_segmenter(sam2_cfg, device)
except ImportError as exc:
    print(exc)

try:  # and a config that asks for SAM 2 without a checkpoint is rejected outright
    SegConfig(data_root=str(ROOT / "seg"), model="sam2")
except ValueError as exc:
    print("\n", exc, sep="")

## 5. Train

In [ ]:
from src.common.seed import make_generator, seed_worker
from src.segmentation.sam2_lora import forward_batch
from src.segmentation.train_sam2 import train_epoch

train_loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=True,
                          collate_fn=collate, generator=make_generator(cfg.seed),
                          worker_init_fn=seed_worker)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate)

opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                        lr=cfg.lr, weight_decay=cfg.weight_decay)

max_steps = cfg.epochs * len(train_loader)
step, losses = 0, []
for epoch in range(cfg.epochs):
    step, hist = train_epoch(model, train_loader, opt, cfg, device, step, max_steps)
    losses += [h["loss"] for h in hist]
    print(f"epoch {epoch:02d} | {model.backend} | train loss {np.mean([h['loss'] for h in hist]):.4f}")

plt.figure(figsize=(6, 3.2))
plt.plot(losses); plt.xlabel("optimizer step"); plt.ylabel("Dice + BCE")
plt.title(f"{model.backend} training loss"); plt.grid(alpha=0.3)
plt.show()

## 6. Score

In [ ]:
from src.common.metrics import dice_iou, merge_scores

@torch.no_grad()
def score(loader, model):
    model.eval()
    return merge_scores(dice_iou(forward_batch(model, b, device).cpu(), b["mask"])
                        for b in loader)

print(f"{model.backend} | {score(train_loader, model).describe('train')}")
print(f"{model.backend} | {score(val_loader, model).describe('val')}   <- selection split")

## 7. Predictions on held-out clips

In [ ]:
test_ds = MaskDataset(str(ROOT / "seg"), split=splits["test"], img_size=64,
                      prompt_kind="point", seed=0, name="test")
test_loader = DataLoader(test_ds, batch_size=cfg.batch_size, shuffle=False, collate_fn=collate)
test_score = score(test_loader, model)
print(f"\n{model.backend} | {test_score.describe('test')} — held out from both "
      "training and selection")

rows = [0, len(test_ds) // 3, 2 * len(test_ds) // 3]
fig, axes = plt.subplots(len(rows), 3, figsize=(9, 3 * len(rows)))
model.eval()
with torch.no_grad():
    for r, idx in enumerate(rows):
        batch = collate([test_ds[idx]])
        pred = (forward_batch(model, batch, device).cpu()[0, 0] > 0).numpy()
        axes[r, 0].imshow(to_display(batch["image"][0]))
        axes[r, 1].imshow(batch["mask"][0, 0], cmap="gray")
        axes[r, 2].imshow(pred, cmap="gray")
        axes[r, 0].set_ylabel(batch["pair_id"][0], fontsize=7)
        for c, t in enumerate(("image", "ground truth", "prediction")):
            axes[r, c].set_xticks([]); axes[r, c].set_yticks([])
            if r == 0:
                axes[r, c].set_title(t, fontsize=10)
plt.tight_layout()
plt.show()

## 8. Save the adapters, and prove the reload is exact

The checkpoint stores trainable not just `lora_A`/`lora_B`.

In [ ]:
from src.common.lora import adapter_state_dict, load_adapter_state_dict

payload = adapter_state_dict(model.adapter_root, model.lora_spec, root_name=model.backend)
print(f"saved {len(payload['tensors'])} tensors:")
for name, t in payload["tensors"].items():
    print(f"  {name:24s} {tuple(t.shape)}")

print(f"\ntrained      | {model.backend} | {score(test_loader, model).describe('test')}")
fresh = build_segmenter(cfg, device)  # an identical architecture, no adapters applied
print(f"fresh init   | {fresh.backend} | {score(test_loader, fresh).describe('test')}")
print("load report:", load_adapter_state_dict(fresh.adapter_root, payload).describe())
print(f"after reload | {fresh.backend} | {score(test_loader, fresh).describe('test')}")